# UCS Satellite Database - Veri Temizleme ve Keşifsel Analiz

**Çalışma Sorusu:** Ticari şirketlerin uzay yarışına katılmasıyla, Dünya yörüngesindeki aktif uydu ve uzay çöpü yoğunluğu son 20 yılda nasıl bir tehlike boyutuna ulaştı?

## 1. Veriyi Yükleme
İlk adım olarak, `data/raw/` klasöründeki Excel dosyamızı Pandas kütüphanesi ile içeri aktarıyoruz.

In [4]:
import pandas as pd

df = pd.read_excel('../data/raw/UCS-Satellite-Database 5-1-2023.xlsx')

#Sütunları gösterek ve ilk 5 satırı kontrol etmek için:
display(df.head())
print(df.info())

,"Name of Satellite, Alternate Names",Current Official Name of Satellite,Country/Org of UN Registry,Country of Operator/Owner,Operator/Owner,Users,Purpose,Detailed Purpose,Class of Orbit,Type of Orbit,...,Unnamed: 58,Unnamed: 59,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67
0,1HOPSAT-TD (1st-generation High Optical Perfor...,1HOPSAT-TD,NR,USA,Hera Systems,Commercial,Earth Observation,Infrared Imaging,LEO,Non-Polar Inclined,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAC AIS-Sat1 (Kelpie 1),AAC AIS-Sat1 (Kelpie 1),United Kingdom,United Kingdom,AAC Clyde Space,Commercial,Earth Observation,Automatic Identification System (AIS),LEO,Sun-Synchronous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Aalto-1,Aalto-1,Finland,Finland,Aalto University,Civil,Technology Development,NaN,LEO,Sun-Synchronous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAt-4,AAt-4,Denmark,Denmark,University of Aalborg,Civil,Earth Observation,Automatic Identification System (AIS),LEO,Sun-Synchronous,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"ABS-2 (Koreasat-8, ST-3)",ABS-2,NR,Multinational,Asia Broadcast Satellite Ltd.,Commercial,Communications,NaN,GEO,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7560 entries, 0 to 7559
Data columns (total 68 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Name of Satellite, Alternate Names  7560 non-null   object 
 1   Current Official Name of Satellite  7560 non-null   object 
 2   Country/Org of UN Registry          7559 non-null   object 
 3   Country of Operator/Owner           7560 non-null   object 
 4   Operator/Owner                      7560 non-null   object 
 5   Users                               7560 non-null   object 
 6   Purpose                             7560 non-null   object 
 7   Detailed Purpose                    1254 non-null   object 
 8   Class of Orbit                      7560 non-null   object 
 9   Type of Orbit                       6909 non-null   object 
 10  Longitude of GEO (degrees)          7557 non-null   float64
 11  Perigee (km)                        7553 no

## 2. Veri Temizleme ve Sütun Seçimi
Orijinal veri setinde 68 sütun bulunuyor ancak hikayemiz için sadece fırlatma tarihi, kullanım aracı, kullanım türü (ticari/askeri) ve yörünge sınıfı (LEO, GEO) gibi temel değişkenlere ihtiyacımız var.
- İlgili olmayan sütunlar ve tamamen boş (NaN) olan 'Unnamed' sütunları veri setinden çıkarıldı.
- Tarih analizi yapabilmek için fırlatma tarihi verisi standart `datetime` formatına dönüştürüldü.

In [5]:
# 1. Hikayemiz için gerekli olan temel sütunları seçiyoruz.
secili_sutunlar = [
    'Name of Satellite, Alternate Names',
    'Country of Operator/Owner',
    'Users', # Ticari, Askeri, Sivil vb.
    'Purpose', # İletişim, Gözlem, vb.
    'Class of Orbit', # LEO, GEO, vb.
    'Date of Launch', # Fırlatma tarihi
]
df_temiz = df[secili_sutunlar].copy()

# 2. Sütun isimlerini daha kolay çalışmak için İngilizce ve kısa hale getirme aşaması
df_temiz.columns = ['Name', 'Country', 'Users_Type', 'Purpose', 'Orbit_Class', 'Launch_Date']

# 3. Tarih sütununu gerçek tarih formatına çevirme (hatalı tarihleri boş bırakıyorum)
df_temiz['Launch_Date'] = pd.to_datetime(df_temiz['Launch_Date'], errors='coerce')

# 4. Fırlatma tarihi belli olmayan (boş olan) uyguları silme
df_temiz = df_temiz.dropna(subset=['Launch_Date'])

# 5. Yıllara göre analiz yapabilmek için sadece fırlatma yılını içeren yeni bir sütun ekleme
df_temiz['Launch_Year'] = df_temiz['Launch_Date'].dt.year

# Sonuç çıktısını kontrol etme
print(f"Temizlenmiş veri seti: {df_temiz.shape[0]} satır, {df_temiz.shape[1]} sütun")
display(df_temiz.head())

Temizlenmiş veri seti: 7557 satır, 7 sütun


,Name,Country,Users_Type,Purpose,Orbit_Class,Launch_Date,Launch_Year
0,1HOPSAT-TD (1st-generation High Optical Perfor...,USA,Commercial,Earth Observation,LEO,2019-12-11,2019
1,AAC AIS-Sat1 (Kelpie 1),United Kingdom,Commercial,Earth Observation,LEO,2023-01-03,2023
2,Aalto-1,Finland,Civil,Technology Development,LEO,2017-06-23,2017
3,AAt-4,Denmark,Civil,Earth Observation,LEO,2016-04-25,2016
4,"ABS-2 (Koreasat-8, ST-3)",Multinational,Commercial,Communications,GEO,2014-02-06,2014


## 3. Keşifsel Veri Analizi ve Hikaye Bulguları
Verimizi temizledikten sonra, scrollytelling sitemde (veri hikayemde) anlatacağım ana bulguları çıkarmak için gruplama işlemlerini yapıyorum.
Hikayemin odak noktası "Ticari şirketlerin uzay yarışındaki yükselişi" olduğu için:
1. Yıllara göre fırlatılan toplam uydu sayısındaki genel eğimi hesaplıyorum.
2. Uyduları kullanıcı türüne göre (Ticari vs. Diğer) ayırıp, ticari uyduların (örneğin SpaceX, Starlink gibi şirketler) son yıllardaki dramatik artışını kıyaslıyorum. 

In [8]:
# 1. Yıllara göre toplam fırlatılan uydu sayısı (Genel artış trendi)
yillik_uydu_sayisi = df_temiz.groupby('Launch_Year').size().reset_index(name='Total_Satellites')

# 2. Ticari (Commercial) ve diğer uyduların yıllara göre kıyaslaması
# 'Users' sütununda 'Commercial' kelimesi geçiyorsa 'Ticari', yoksa 'Diğer' olarak etiketliyoruz
df_temiz['Users_Category'] = df_temiz['Users_Type'].apply(lambda x: 'Ticari' if 'Commercial' in str(x) else 'Diğer')
kategori_yillik = df_temiz.groupby(['Launch_Year', 'Users_Category']).size().unstack(fill_value=0).reset_index()

# Çıktıları ekrana yazdırma
print("Yıllara göre toplam uydu artışı (Son 10 yıl):")
display(yillik_uydu_sayisi.tail(10))

print("\nTicari vs Diğer uyduların fırlatma sayıları (Son 10 yıl):")
display(kategori_yillik.tail(10))


Yıllara göre toplam uydu artışı (Son 10 yıl):


,Launch_Year,Total_Satellites
27,2014,117
28,2015,132
29,2016,107
30,2017,221
31,2018,308
32,2019,272
33,2020,1054
34,2021,1590
35,2022,2113
36,2023,909



Ticari vs Diğer uyduların fırlatma sayıları (Son 10 yıl):


Users_Category,Launch_Year,Diğer,Ticari
27,2014,62,55
28,2015,69,63
29,2016,65,42
30,2017,69,152
31,2018,140,168
32,2019,90,182
33,2020,97,957
34,2021,153,1437
35,2022,100,2013
36,2023,53,856


## 4. Sitede Kullanılacak Özet Dosyaların Üretimi
Bölüm 7.6 (Performans) gerekliliklerine uymak adına, web sitemde ham veri yerine yalnızca analiz sonucu elde ettiğim özet tabloları kullanacağım. Bu nedenle, hikayemin akışını oluşturacak olan 'yıllık uydu sayısı' ve 'ticari vs diğer uydu' kırılımı verilerini `data/processed/` klasörüne CSV formatında dışa aktarıyorum.

In [9]:
# Verileri web sitesi klasörüne (processed) aktarma
import os

# Processsed klasörünün var olduğundan emin olma işlemi
os.makedirs('../data/processed', exist_ok=True)

#CSV olarak kaydetme
yillik_uydu_sayisi.to_csv('../data/processed/yillik_uydu_sayisi.csv', index=False)
kategori_yillik.to_csv('../data/processed/ticari_vs_diger.csv', index=False)

print("Özet dosyalar başarıyla data/processed/ klasörüne kaydedildi!")

Özet dosyalar başarıyla data/processed/ klasörüne kaydedildi!
